# Optically Shallow / Deep Water Delineation

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/optically_shallow_deep.ipynb)

This notebook demonstrates how to perform optically shallow and deep water classification in Sentinel-2 imagery using opticallyshallowdeep and the geoai library.

## Install Package

Ensure geoai and opticallyshallowdeep are installed in your environment.

In [ ]:
%pip install "geoai-py[osd]"

### A. Classification for Level-1C (L1C) SAFE Files

In [ ]:
import geoai
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# 1. Define file paths
# Path to Sentinel-2 Level-1C SAFE directory
file_L1C = "S2B_MSIL1C_20190725T100039_N0208_R122_T33UWP_20190725T123957.SAFE"

# Output probability GeoTIFF path
out_prob = "osd_probability.tif"

# 2. Run classification
geoai.classify_optically_shallow_deep(
    image=file_L1C, output=out_prob, to_log=True  # False
)

In [ ]:
# Derive TCI (True Color Image) path from SAFE directory structure
tci_matches = list(Path(file_L1C).glob("GRANULE/*/IMG_DATA/*TCI*.jp2"))
tci_path = (
    str(tci_matches[0])
    if tci_matches
    else f"{file_L1C}/GRANULE/L1C_T33UWP_A012446_20190725T100415/IMG_DATA/T33UWP_20190725T100039_TCI.jp2"
)

# 3. Read and downsample raster data for visualization
with rasterio.open(tci_path) as src:
    h, w = src.height // 5, src.width // 5
    rgb = src.read([1, 2, 3], out_shape=(3, h, w), resampling=Resampling.bilinear)
    rgb = np.transpose(rgb, (1, 2, 0))
with rasterio.open(out_prob) as src:
    prob = src.read(1, out_shape=(1, h, w), resampling=Resampling.nearest)
    prob_masked = np.ma.masked_where(prob == 255, prob)

# 4. Plot TCI Scene and OSW Probability Map side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left: True Color Image (TCI)
axes[0].imshow(rgb)
axes[0].set_title("Sentinel-2 Scene (TCI)", fontsize=14, pad=10)
axes[0].axis("off")

# Right: OSW Probability Map
im = axes[1].imshow(prob_masked, cmap="YlOrRd", vmin=0, vmax=100)
axes[1].set_title("Optically Shallow Water (OSW) Probability Map", fontsize=14, pad=10)
axes[1].axis("off")

# Add colorbar
cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("OSW Probability (%)", fontsize=12)
cbar.set_ticks([0, 25, 50, 75, 100])
plt.tight_layout()
plt.show()

### B. Classification for ACOLITE Level-2R (L2R) Files

For ACOLITE Level-2R (surface reflectance) NetCDF files, the original Level-1C SAFE directory is also provided to extract the built-in cloud mask.

In [ ]:
import geoai

# Input Level-1C SAFE file (for cloud mask)
file_L1C = "S2B_MSIL1C_20190725T100039_N0208_R122_T33UWP_20190725T123957.SAFE"

# ACOLITE Level-2R NetCDF file matching the same scene
file_L2R = "S2B_MSI_2019_07_25_10_00_39_T33UWP_L2R.nc"

# Output probability GeoTIFF path
out_prob = "osd_probability_l2r.tif"

# Run classification with surface reflectance
geoai.classify_optically_shallow_deep(
    image=file_L1C, output=out_prob, acolite_l2r=file_L2R, to_log=True
)

### Interactive Visualization with Leafmap

In [ ]:
import leafmap

m = leafmap.Map()
m.add_basemap("HYBRID")
m.add_raster(out_prob, cmap="viridis", layer_name="OSW Probability")
m.fit_bounds(leafmap.image_bounds(out_prob))
m